# Loop 01 — the hand-labeled anchor: verification & blind sheet

Executable detail for steps 3–4 of the labeling protocol (see this loop's
`README.md`): resolve the researcher's recall-first entries (`recall-raw.csv`)
against the local S2 corpus, verify each recalled citer actually cites its
seed (the hit-rate is part (b) of the hypothesis), then draw a uniform random
sample of each seed's remaining citers for blind labeling.

**The blind rule:** this notebook never *displays* a citation count. Counts
are touched in exactly one internal place — picking the canonical record when
the corpus holds duplicate rows for the same paper — and never surface in any
output or in the labeling sheet.

In [1]:
"""Verify recalled citers against the corpus and draw the blind labeling sample."""

import csv
import hashlib
import json
from pathlib import Path

import duckdb

from atlas.integrations.semantic_scholar.corpus import paths
from atlas.integrations.semantic_scholar.corpus.ingest import NBUCKETS

corpus_root = paths.corpus_root()
release_id = paths.read_current_release(corpus_root)
release = paths.release_paths(release_id)
papers_glob = (release.parquet_dataset("papers") / "*.parquet").as_posix()
citations_root = release.parquet_dataset("citations").as_posix()

connection = duckdb.connect()
print(f"corpus release: {release_id}")

corpus release: 2026-07-07


## Resolving the recalled entries

`recall-raw.csv` names papers the way a person remembers them ("AlexNet",
"YOLO"), so each entry is mapped to its exact corpus title plus its
publication year — the year disambiguates generic titles (three corpus papers
are titled "Q-learning"; Hawking's 1974 *Black hole explosions?* has a 1982
namesake). All resolutions were confirmed by the researcher on 2026-07-25,
including the call that "Visual Transformers" means the 16×16-words ViT paper.

In [2]:
# Display name (as recalled) -> (exact corpus title, lowered; publication year).
SEED_SPEC = {
    "Attention Is All You Need": ("attention is all you need", 2017),
    "Playing Atari with Deep Reinforcement Learning": ("playing atari with deep reinforcement learning", 2013),
    "Correlated Q-Learning": ("correlated q-learning", 2003),
    "AlexNet": ("imagenet classification with deep convolutional neural networks", 2012),
    "Black Hole Explosions?": ("black hole explosions?", 1974),
    "Q-Learning": ("q-learning", 1992),
    "Long Short-Term Memory": ("long short-term memory", 1997),
    "YOLO": ("you only look once: unified, real-time object detection", 2015),
    "ViT": ("an image is worth 16x16 words: transformers for image recognition at scale", 2020),
    "QMIX": ("qmix: monotonic value function factorisation for deep multi-agent reinforcement learning", 2018),
}

# Seed display name -> {recalled citer display name -> (lowered title, year)}.
RECALL_SPEC = {
    "Attention Is All You Need": {
        "LoRA": ("lora: low-rank adaptation of large language models", 2021),
        "Sparsely-Gated MoE": ("outrageously large neural networks: the sparsely-gated mixture-of-experts layer", 2017),
        "BERT": ("bert: pre-training of deep bidirectional transformers for language understanding", 2019),
    },
    "Playing Atari with Deep Reinforcement Learning": {
        "DDPG": ("continuous control with deep reinforcement learning", 2015),
        "PPO": ("proximal policy optimization algorithms", 2017),
        "Double DQN": ("deep reinforcement learning with double q-learning", 2015),
        "Rainbow": ("rainbow: combining improvements in deep reinforcement learning", 2017),
    },
    "AlexNet": {
        "ResNet": ("deep residual learning for image recognition", 2015),
        "Attention Is All You Need": ("attention is all you need", 2017),
    },
}


def resolve_entries(spec_entries: dict[str, tuple[str, int]]) -> dict[str, int]:
    """Resolve (lowered title, year) pairs to corpusids in one papers scan.

    Among rows sharing the same lowered title, the publication year picks the
    intended paper; if the corpus still holds duplicate records for it, the
    most-cited record is taken as the canonical one (an identity decision made
    internally — the count is never displayed).

    Args:
        spec_entries: Display name -> (exact title lowered, publication year).

    Returns:
        Display name -> resolved corpusid.
    """
    wanted_titles = sorted({title for title, _ in spec_entries.values()})
    placeholders = ", ".join("?" for _ in wanted_titles)
    rows = connection.execute(
        f"SELECT lower(title), corpusid, year, citationcount "
        f"FROM read_parquet('{papers_glob}') "
        f"WHERE lower(title) IN ({placeholders})",
        wanted_titles,
    ).fetchall()
    resolved: dict[str, int] = {}
    for display_name, (wanted_title, wanted_year) in spec_entries.items():
        candidates = [
            (corpusid, citationcount or 0)
            for lowered, corpusid, year, citationcount in rows
            if lowered == wanted_title and year == wanted_year
        ]
        if not candidates:
            raise ValueError(f"no corpus match for {display_name!r} ({wanted_title!r}, {wanted_year})")
        # Canonical-record pick among duplicates; the count itself stays internal.
        resolved[display_name] = max(candidates, key=lambda candidate: candidate[1])[0]
    return resolved


all_recalled_spec = {
    f"{seed_name} <- {citer_name}": citer_entry
    for seed_name, citer_entries in RECALL_SPEC.items()
    for citer_name, citer_entry in citer_entries.items()
}
seed_ids = resolve_entries(SEED_SPEC)
recalled_ids = resolve_entries(all_recalled_spec)

print("seeds:")
for seed_name, seed_corpusid in seed_ids.items():
    print(f"  {seed_name}: corpusid={seed_corpusid}")
print("recalled citers:")
for recall_key, citer_corpusid in recalled_ids.items():
    print(f"  {recall_key}: corpusid={citer_corpusid}")

seeds:
  Attention Is All You Need: corpusid=13756489
  Playing Atari with Deep Reinforcement Learning: corpusid=15238391
  Correlated Q-Learning: corpusid=5228823
  AlexNet: corpusid=195908774
  Black Hole Explosions?: corpusid=4290107
  Q-Learning: corpusid=208910339
  Long Short-Term Memory: corpusid=1915014
  YOLO: corpusid=206594738
  ViT: corpusid=225039882
  QMIX: corpusid=4533648
recalled citers:
  Attention Is All You Need <- LoRA: corpusid=235458009
  Attention Is All You Need <- Sparsely-Gated MoE: corpusid=12462234
  Attention Is All You Need <- BERT: corpusid=52967399
  Playing Atari with Deep Reinforcement Learning <- DDPG: corpusid=16326763
  Playing Atari with Deep Reinforcement Learning <- PPO: corpusid=28695052
  Playing Atari with Deep Reinforcement Learning <- Double DQN: corpusid=6208256
  Playing Atari with Deep Reinforcement Learning <- Rainbow: corpusid=19135734
  AlexNet <- ResNet: corpusid=206594692
  AlexNet <- Attention Is All You Need: corpusid=13756489


## Verification — are the recalled landmarks actually citers? (hypothesis part b)

For each seed, pull its full set of *distinct* citing papers and check every
recalled name for membership. The edge list ships every edge about twice
(overlapping S2 export batches — see `corpus/README.md`), so the query groups
by the citing paper before anything counts it.

A miss is as informative as a hit: it means human memory of "a landmark
downstream of this paper" doesn't correspond to a direct citation edge —
either the memory is wrong, the corpus is missing the edge, or influence
flowed without a direct citation.

In [3]:
def distinct_citers(seed_corpusid: int) -> set[int]:
    """Every distinct citing corpusid of a seed, from its hash bucket.

    Args:
        seed_corpusid: The seed's corpusid; also selects the edge bucket
            (edges are hash-partitioned on ``citedcorpusid % NBUCKETS``).

    Returns:
        The set of citing corpusids, deduplicated across S2's overlapping
        export batches.
    """
    bucket = seed_corpusid % NBUCKETS
    citations_glob = f"{citations_root}/bucket={bucket}/*.parquet"
    rows = connection.execute(
        f"SELECT citingcorpusid FROM read_parquet('{citations_glob}', hive_partitioning=false) "
        "WHERE citedcorpusid = ? GROUP BY citingcorpusid",
        [seed_corpusid],
    ).fetchall()
    return {int(row[0]) for row in rows}


citer_pools = {seed_name: distinct_citers(seed_corpusid) for seed_name, seed_corpusid in seed_ids.items()}

print("citer pool sizes (the label targets' own counts stay hidden — these are")
print("pool sizes, shown because the spread itself is a finding):")
for seed_name, pool in citer_pools.items():
    print(f"  {seed_name}: {len(pool):,} distinct citers")

hits = 0
misses: list[str] = []
print("\nrecalled-citer verification:")
for seed_name, citer_entries in RECALL_SPEC.items():
    pool = citer_pools[seed_name]
    for citer_name in citer_entries:
        citer_corpusid = recalled_ids[f"{seed_name} <- {citer_name}"]
        in_pool = citer_corpusid in pool
        hits += int(in_pool)
        if not in_pool:
            misses.append(f"{seed_name} <- {citer_name}")
        print(f"  {seed_name} <- {citer_name}: {'IN POOL' if in_pool else 'NOT IN POOL'}")

recall_total = sum(len(citer_entries) for citer_entries in RECALL_SPEC.values())
print(f"\nhit rate: {hits}/{recall_total}")
if misses:
    print("misses to investigate:", "; ".join(misses))

citer pool sizes (the label targets' own counts stay hidden — these are
pool sizes, shown because the spread itself is a finding):
  Attention Is All You Need: 180,306 distinct citers
  Playing Atari with Deep Reinforcement Learning: 13,729 distinct citers
  Correlated Q-Learning: 476 distinct citers
  AlexNet: 129,463 distinct citers
  Black Hole Explosions?: 5,114 distinct citers
  Q-Learning: 12,396 distinct citers
  Long Short-Term Memory: 105,656 distinct citers
  YOLO: 45,812 distinct citers
  ViT: 64,334 distinct citers
  QMIX: 2,009 distinct citers

recalled-citer verification:
  Attention Is All You Need <- LoRA: IN POOL
  Attention Is All You Need <- Sparsely-Gated MoE: NOT IN POOL
  Attention Is All You Need <- BERT: IN POOL
  Playing Atari with Deep Reinforcement Learning <- DDPG: IN POOL
  Playing Atari with Deep Reinforcement Learning <- PPO: NOT IN POOL
  Playing Atari with Deep Reinforcement Learning <- Double DQN: IN POOL
  Playing Atari with Deep Reinforcement Learnin

## The blind sample (step 4)

Up to 15 citers per seed, drawn **uniformly** from the seed's distinct citer
pool with the recalled positives excluded. "Uniform" is implemented as an
ordering by the MD5 hash of the citer's corpusid: deterministic (the draw is
reproducible run-to-run), and independent of every paper property — a hash of
an arbitrary id knows nothing about citations, age, or venue. No top-N by any
metric anywhere.

The sheet carries title, authors, year, and arXiv id only — no counts.

In [4]:
SAMPLE_PER_SEED = 15


def sample_uniform(pool: set[int], exclude: set[int], sample_size: int) -> list[int]:
    """Draw a deterministic, metric-independent sample from a citer pool.

    Args:
        pool: The seed's distinct citing corpusids.
        exclude: Corpusids that must not be drawn (the recalled positives).
        sample_size: How many to draw (fewer if the pool is smaller).

    Returns:
        The sampled corpusids, in their (pseudo-random) hash order.
    """

    def id_hash(corpusid: int) -> str:
        """A stable pseudo-random key for one corpusid.

        Args:
            corpusid: The id to hash.

        Returns:
            The MD5 hex digest of the id's decimal string.
        """
        return hashlib.md5(str(corpusid).encode()).hexdigest()

    return sorted(pool - exclude, key=id_hash)[:sample_size]


def format_authors(authors_json: str | None) -> str:
    """Join the papers dataset's authors JSON into a display string.

    Args:
        authors_json: The raw ``authors`` value — a JSON array of objects
            carrying ``name`` — or None.

    Returns:
        "Name One, Name Two", or "" when no named authors exist.
    """
    if not authors_json:
        return ""
    try:
        author_entries = json.loads(authors_json)
    except (TypeError, ValueError):
        return ""
    names = [
        entry["name"]
        for entry in author_entries
        if isinstance(entry, dict) and entry.get("name")
    ]
    return ", ".join(names)


recalled_by_seed = {
    seed_name: {recalled_ids[f"{seed_name} <- {citer_name}"] for citer_name in citer_entries}
    for seed_name, citer_entries in RECALL_SPEC.items()
}
samples = {
    seed_name: sample_uniform(pool, recalled_by_seed.get(seed_name, set()), SAMPLE_PER_SEED)
    for seed_name, pool in citer_pools.items()
}

sampled_ids = [citer_corpusid for sample in samples.values() for citer_corpusid in sample]
placeholders = ", ".join("?" for _ in sampled_ids)
# Hydrate the sampled citers wide - deliberately WITHOUT citationcount.
hydrated = {
    int(row[0]): row
    for row in connection.execute(
        f"SELECT corpusid, arxiv_id, year, title, authors "
        f"FROM read_parquet('{papers_glob}') WHERE corpusid IN ({placeholders})",
        sampled_ids,
    ).fetchall()
}

sheet_path = Path("labeling-sheet.csv")
if sheet_path.exists():
    # NEVER overwrite: once the researcher has filled the label column, this
    # file is primary data - a rerun of the notebook must not clobber it. The
    # draw is deterministic (hash order), so the rows are the same anyway.
    print(f"{sheet_path} already exists - leaving the researcher's labels untouched")
else:
    with open(sheet_path, "w", newline="", encoding="utf-8") as sheet_file:
        writer = csv.writer(sheet_file)
        writer.writerow(["seed", "citer_corpusid", "arxiv_id", "year", "title", "authors", "label", "notes"])
        for seed_name, sample in samples.items():
            for citer_corpusid in sample:
                corpusid, arxiv_id, year, title, authors_json = hydrated[citer_corpusid]
                writer.writerow([
                    seed_name, corpusid, arxiv_id or "", year or "",
                    title or "", format_authors(authors_json), "", "",
                ])
    print(f"wrote {sum(len(sample) for sample in samples.values())} rows to {sheet_path}")

for seed_name, sample in samples.items():
    print(f"{seed_name}: {len(sample)} sampled")

labeling-sheet.csv already exists - leaving the researcher's labels untouched
Attention Is All You Need: 15 sampled
Playing Atari with Deep Reinforcement Learning: 15 sampled
Correlated Q-Learning: 15 sampled
AlexNet: 15 sampled
Black Hole Explosions?: 15 sampled
Q-Learning: 15 sampled
Long Short-Term Memory: 15 sampled
YOLO: 15 sampled
ViT: 15 sampled
QMIX: 15 sampled


## The anchor dataset

Assembles `anchor.csv` — the workstream's ground truth — from the two
provenances: corpus-verified recall positives (`landmark`, provenance
`recalled`) and the researcher's blind-sheet labels (provenance `sampled`).
The provenance column matters because the two kinds of row were produced
under different conditions: recalled positives from memory before any list
was shown, sampled labels from recognition of a uniform draw.

In [5]:
# Provenance of the labels this cell consumes: supplied by the researcher on
# 2026-07-25 - they reviewed all 150 sampled rows and recognized none, so per
# the pinned label semantics the 135 within-field rows read `not` and the 15
# rows under the out-of-field seed (Hawking's citers) read `unsure`.
OUT_OF_FIELD_SEEDS = {"Black Hole Explosions?"}

anchor_rows: list[dict[str, object]] = []

# The corpus-verified recall positives enter as `landmark`; the three recall
# misses are excluded (they are not citers - see the README's results).
for seed_name, citer_entries in RECALL_SPEC.items():
    pool = citer_pools[seed_name]
    for citer_name in citer_entries:
        citer_corpusid = recalled_ids[f"{seed_name} <- {citer_name}"]
        if citer_corpusid in pool:
            anchor_rows.append({
                "seed": seed_name,
                "citer_corpusid": citer_corpusid,
                "label": "landmark",
                "provenance": "recalled",
            })

# The blind-sheet labels enter under `sampled` provenance; an unlabeled row
# means the sheet isn't finished and the anchor must not be built from it.
with open(sheet_path, encoding="utf-8", newline="") as sheet_file:
    for sheet_row in csv.DictReader(sheet_file):
        if not sheet_row["label"]:
            raise ValueError(f"unlabeled sheet row: {sheet_row['citer_corpusid']}")
        anchor_rows.append({
            "seed": sheet_row["seed"],
            "citer_corpusid": int(sheet_row["citer_corpusid"]),
            "label": sheet_row["label"],
            "provenance": "sampled",
        })

anchor_path = Path("anchor.csv")
with open(anchor_path, "w", newline="", encoding="utf-8") as anchor_file:
    writer = csv.DictWriter(anchor_file, fieldnames=["seed", "citer_corpusid", "label", "provenance"])
    writer.writeheader()
    writer.writerows(anchor_rows)

label_counts: dict[str, int] = {}
for anchor_row in anchor_rows:
    label_counts[str(anchor_row["label"])] = label_counts.get(str(anchor_row["label"]), 0) + 1
print(f"anchor written to {anchor_path}: {len(anchor_rows)} rows")
for label_name, label_count in sorted(label_counts.items()):
    print(f"  {label_name}: {label_count}")

anchor written to anchor.csv: 156 rows
  landmark: 6
  not: 135
  unsure: 15


## What happens next

The researcher fills the sheet's `label` column — `landmark` / `not` /
`unsure` — from recognition only (no Google, no Semantic Scholar, no citation
counts; the arXiv abstract page is safe if a title is unfamiliar). The unsure
rate is part (a) of the hypothesis. The filled sheet plus the verified recall
positives become the anchor dataset, and the interpretation lands in this
loop's README.